# L3 Photonic Crystal Cavity

An L3 cavity is formed by removing 3 adjacent holes from a hexagonal lattice photonic crystal.

In [ ]:
import legume
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle

print("✓ Legume imported successfully!")

## Setup: Hexagonal Lattice Parameters

In [ ]:
# Lattice constant
a = 1.0

# Hexagonal lattice vectors
lattice = legume.Lattice([a, 0], [a/2, a*np.sqrt(3)/2])

# Hole radius (typically r = 0.3*a for good bandgap)
r = 0.3 * a

print(f"Lattice constant: {a}")
print(f"Hole radius: {r}")

## Create Photonic Crystal Slab

In [ ]:
# Create photonic crystal
phc = legume.PhotCryst(lattice)

# Add slab layer (GaAs: eps ~ 12)
d_slab = 0.5 * a  # Slab thickness
eps_slab = 12.0   # Permittivity of GaAs

phc.add_layer(d=d_slab, eps_b=eps_slab)

print(f"Slab thickness: {d_slab}")
print(f"Slab permittivity: {eps_slab}")

## Add Hexagonal Hole Array with L3 Defect

In [ ]:
# Define supercell size (larger to accommodate L3 cavity)
nx, ny = 10, 8  # Number of holes in x and y directions

# Add holes in hexagonal pattern
for i in range(-nx, nx+1):
    for j in range(-ny, ny+1):
        # Position of hole
        x = i * a + (j % 2) * a/2
        y = j * a * np.sqrt(3)/2
        
        # Skip 3 holes in center to create L3 cavity
        # Remove holes at (0,0), (1,0), (2,0) in lattice coordinates
        if (i == 0 and j == 0) or (i == 1 and j == 0) or (i == 2 and j == 0):
            continue
        
        # Add circular hole (air: eps = 1)
        phc.layers[0].add_shape(legume.Circle(x_cent=x, y_cent=y, r=r, eps=1.0))

print(f"✓ L3 cavity created: 3 missing holes along x-direction")
print(f"Total shapes in layer: {len(phc.layers[0].shapes)}")

## Visualize the Structure

In [ ]:
# Plot the photonic crystal structure
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

# Plot each hole
for shape in phc.layers[0].shapes:
    if hasattr(shape, 'r'):  # Circle
        circle = Circle((shape.x_cent, shape.y_cent), shape.r, 
                       color='lightblue', ec='blue', linewidth=0.5, alpha=0.7)
        ax.add_patch(circle)

# Highlight the cavity region
cavity_box = Rectangle((-0.2, -0.8), 2.5, 1.6, 
                      linewidth=3, edgecolor='red', 
                      facecolor='none', linestyle='--',
                      label='L3 Cavity Region')
ax.add_patch(cavity_box)

ax.set_xlim(-6, 6)
ax.set_ylim(-5, 5)
ax.set_xlabel('x/a', fontsize=12)
ax.set_ylabel('y/a', fontsize=12)
ax.set_title('L3 Photonic Crystal Cavity (Top View)', fontsize=14)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

## Cross-Section View

In [ ]:
# Visualize permittivity profile along a line
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Top view with cut line
for shape in phc.layers[0].shapes:
    if hasattr(shape, 'r'):
        circle = Circle((shape.x_cent, shape.y_cent), shape.r, 
                       color='lightblue', ec='blue', linewidth=0.5, alpha=0.7)
        ax1.add_patch(circle)

ax1.axhline(y=0, color='red', linewidth=2, linestyle='--', label='Cross-section line')
ax1.set_xlim(-6, 6)
ax1.set_ylim(-5, 5)
ax1.set_xlabel('x/a')
ax1.set_ylabel('y/a')
ax1.set_title('Top View - L3 Cavity')
ax1.legend()
ax1.set_aspect('equal')
ax1.grid(True, alpha=0.3)

# Cross-section along y=0
x_line = np.linspace(-5, 5, 500)
eps_line = []

for x_val in x_line:
    # Check if point is inside any hole
    in_hole = False
    for shape in phc.layers[0].shapes:
        if hasattr(shape, 'r'):  # Circle
            dist = np.sqrt((x_val - shape.x_cent)**2 + (0 - shape.y_cent)**2)
            if dist < shape.r:
                in_hole = True
                break
    
    eps_line.append(1.0 if in_hole else eps_slab)

ax2.plot(x_line, eps_line, linewidth=2, color='darkblue')
ax2.fill_between(x_line, 0, eps_line, alpha=0.3, color='lightblue')
ax2.set_xlabel('x/a', fontsize=12)
ax2.set_ylabel('Permittivity ε', fontsize=12)
ax2.set_title('Cross-section: Permittivity Profile at y=0', fontsize=12)
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0, 13])

# Mark cavity region
ax2.axvspan(0, 2, alpha=0.2, color='red', label='L3 Cavity')
ax2.legend()

plt.tight_layout()
plt.show()

## Zoom-in on L3 Cavity

In [ ]:
# Close-up view of the L3 cavity region
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Plot holes near cavity
for shape in phc.layers[0].shapes:
    if hasattr(shape, 'r'):
        # Only plot if near cavity region
        if abs(shape.x_cent) < 4 and abs(shape.y_cent) < 3:
            circle = Circle((shape.x_cent, shape.y_cent), shape.r, 
                           color='lightblue', ec='blue', linewidth=1, alpha=0.7)
            ax.add_patch(circle)
            # Add labels for nearest neighbors
            if abs(shape.x_cent) < 2 and abs(shape.y_cent) < 1.5:
                ax.plot(shape.x_cent, shape.y_cent, 'r.', markersize=5)

# Mark the missing holes (L3 cavity)
for i in range(3):
    x_missing = i * a
    y_missing = 0
    missing = Circle((x_missing, y_missing), r, 
                    color='none', ec='red', linewidth=2, linestyle='--',
                    label='Missing holes' if i==0 else '')
    ax.add_patch(missing)
    ax.text(x_missing, y_missing, 'X', ha='center', va='center', 
           fontsize=16, color='red', fontweight='bold')

ax.set_xlim(-2, 4)
ax.set_ylim(-2, 2)
ax.set_xlabel('x/a', fontsize=12)
ax.set_ylabel('y/a', fontsize=12)
ax.set_title('L3 Cavity - Close-up View', fontsize=14)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

## Summary

**What we created:**
- Hexagonal lattice photonic crystal (mimicking GaAs membrane)
- L3 cavity: 3 missing holes in a row along x-direction
- This cavity can trap light with high Q-factor

**Key Parameters:**
- Lattice constant: a = 1.0 (normalized)
- Hole radius: r = 0.3a
- Slab thickness: d = 0.5a
- Slab permittivity: ε = 12 (GaAs)

**Next Steps:**
- Run guided mode expansion to find cavity modes
- Calculate Q-factor and mode volume
- Optimize cavity geometry (shift outer holes, adjust radius)

## Optional: Compute Guided Modes

To find the electromagnetic modes, you can run GME simulation:

In [ ]:
# This is just a template - GME computation takes time
# Uncomment to run actual simulation:

# # Create GME object
# gmax = 5  # Number of G-vectors
# gme = legume.GuidedModeExp(phc, gmax=gmax)
# 
# # Run at Gamma point (k=0)
# gme.run(kpoints=np.array([[0, 0]]))
# 
# # Get frequencies
# freqs = gme.freqs
# print(f"Frequencies at Γ point: {freqs[0, :5]}")